# 01 Model Development

**Author:** Rowan Walker

This notebook contains code that was used to design, test and iterate the model including all stages from data pipeline to training, inference and back testing. Multiple iterations have been performed over the course of model development, however this notebook is designed to show the steps that an end-to-end run entail.

### 1.1 Imports

In [14]:
import numpy as np
import pandas as pd

from src.data.preprocess import generate_valid_tickers, generate_model_inputs
from src.inference.inference import model_inference
from src.training.train import train_model
from src.utils.backtest import backtest, optimise_sharpe
from src.utils.risk import full_risk_report
from src.utils.seed import set_global_seed

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torch")
warnings.filterwarnings("ignore", category=FutureWarning, module="torch")
warnings.filterwarnings("ignore", category=UserWarning, module="hmmlearn")

%reload_ext autoreload
%autoreload 2

# Global seed setting for reproducibility
set_global_seed()

### 1.2 Testing prediction windows and model training

In [3]:
for i, t in enumerate((1, 5, 21)):
    valid_tickers = generate_valid_tickers(start_date='2010-12-01',
                                           end_date='2025-12-01')
    
    X, y, stock_ids, regime_X, full_df = generate_model_inputs(tickers=valid_tickers,
                                                               train_start='2010-12-01',
                                                               train_end='2020-12-01',
                                                               hold_days=t)
    print('\n')
    train_model(X, y, stock_ids, regime_X, hold_days=t)

    if i<2:
        print('\nNext iteration:\n')

Load successful
----------------
Time taken: 4.64s
Samples: 11385
Features: 28
Stocks: 5




/Users/rowanwalker/rowan-env-312/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Beginning training for 5 epochs on device "mps"
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Epoch 1: Loss = 0.6963. Time taken = 8.61s
Epoch 2: Loss = 0.6950. Time taken = 8.14s
Epoch 3: Loss = 0.6948. Time taken = 8.16s
Epoch 4: Loss = 0.6942. Time taken = 8.19s
Epoch 5: Loss = 0.6937. Time taken = 8.11s
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Training completed. Model saved: /Users/rowanwalker/rowan-env-312/multi-asset-transformer/results/model/model_1.pth

Next iteration:

Load successful
----------------
Time taken: 4.67s
Samples: 11365
Features: 28
Stocks: 5


Beginning training for 5 epochs on device "mps"
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


/Users/rowanwalker/rowan-env-312/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Epoch 1: Loss = 0.6944. Time taken = 9.36s
Epoch 2: Loss = 0.6935. Time taken = 8.73s
Epoch 3: Loss = 0.6924. Time taken = 8.67s
Epoch 4: Loss = 0.6912. Time taken = 8.24s
Epoch 5: Loss = 0.6898. Time taken = 8.24s
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Training completed. Model saved: /Users/rowanwalker/rowan-env-312/multi-asset-transformer/results/model/model_5.pth

Next iteration:

Load successful
----------------
Time taken: 4.55s
Samples: 11285
Features: 28
Stocks: 5


Beginning training for 5 epochs on device "mps"
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


/Users/rowanwalker/rowan-env-312/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Epoch 1: Loss = 0.7012. Time taken = 8.73s
Epoch 2: Loss = 0.6847. Time taken = 8.25s
Epoch 3: Loss = 0.6836. Time taken = 8.64s
Epoch 4: Loss = 0.6823. Time taken = 8.28s
Epoch 5: Loss = 0.6818. Time taken = 8.31s
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Training completed. Model saved: /Users/rowanwalker/rowan-env-312/multi-asset-transformer/results/model/model_21.pth


### 1.3 Inference loop

In [4]:
model_inference(valid_tickers, full_df, feature_dim=X.shape[2], hold_days=(1,5,21))

/Users/rowanwalker/rowan-env-312/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/Users/rowanwalker/rowan-env-312/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Predicted dataframe saved: /Users/rowanwalker/rowan-env-312/multi-asset-transformer/data/inference
Time taken: 9.84 seconds


### 1.4 Sharpe optimisation

In [16]:
param_grid ={'long_threshold': np.arange(55, 80, 5)/100,
             'short_threshold': np.arange(25, 50, 5)/100,
             'target_vol': np.arange(10, 30, 5)/100,
             'slippage': 1,
             'commission': 1,
             'take_profit': np.arange(2, 21)/100,
             'stop_loss': -np.arange(2, 11)/100,
             'max_hold_days': np.arange(15, 31),
             'max_drawdown': 0.2,
             'leverage': np.arange(1, 10)}

param_grid_best, sharpe = optimise_sharpe(param_grid,
                                          trials=300,
                                          start_date='2021-01-01',
                                          end_date='2025-11-01')

Starting optimisation for 300 trials...

Trial 10/300 completed.
Trial 20/300 completed.
Trial 30/300 completed.
Trial 40/300 completed.
Trial 50/300 completed.
Trial 60/300 completed.
Trial 70/300 completed.
Trial 80/300 completed.
Trial 90/300 completed.
Trial 100/300 completed.
Trial 110/300 completed.
Trial 120/300 completed.
Trial 130/300 completed.
Trial 140/300 completed.
Trial 150/300 completed.
Trial 160/300 completed.
Trial 170/300 completed.
Trial 180/300 completed.
Trial 190/300 completed.
Trial 200/300 completed.
Trial 210/300 completed.
Trial 220/300 completed.
Trial 230/300 completed.
Trial 240/300 completed.
Trial 250/300 completed.
Trial 260/300 completed.
Trial 270/300 completed.
Trial 280/300 completed.
Trial 290/300 completed.
Trial 300/300 completed.


In [17]:
param_grid_best, sharpe

({'long_threshold': 0.6,
  'short_threshold': 0.4,
  'target_vol': 0.2,
  'slippage': 1.0,
  'commission': 1.0,
  'take_profit': 0.16,
  'stop_loss': -0.02,
  'max_hold_days': 18.0,
  'max_drawdown': 0.2,
  'leverage': 7.0},
 3.669193134241846)

### 1.5 Back testing

In [20]:
s, b, r = backtest(param_grid_best,
                   start_date='2020-01-01',
                   end_date='2025-11-01')

TypeError: cannot unpack non-iterable numpy.float64 object

### 1.6 Risk Report

In [19]:
full_risk_report(s, b, r)

,Strategy,Benchmark
Mean Daily Return,0.31%,0.03%
Daily Vol,1.30%,1.08%
Annualized Vol,20.64%,17.11%
Sharpe Ratio,3.63,0.21
Sortino Ratio,6.39,0.25
Max Drawdown,-7.70%,-33.72%
Drawdown Days,32,414
Win Rate,59.17%,54.65%
Historical VaR (5%),-1.67%,-1.55%
Gaussian VaR (5%),-1.83%,-1.75%
